In [1]:
print("RAg")

RAg


In [3]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

Using Python 3.11.7 environment at: E:\My_Projects\Intelligent-Document-Assistant-with-OCR-RAG\.venv
Resolved 68 packages in 6.92s
 Downloaded openai
 Downloaded shapely
 Downloaded pydantic-core
 Downloaded sqlalchemy
 Downloaded langchain-community
 Downloaded pillow
 Downloaded numpy
 Downloaded onnxruntime
 Downloaded rapidocr-onnxruntime
 Downloaded opencv-python
Prepared 64 packages in 1m 00s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 64 packages in 7.89s
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.2
 + aiosignal==1.4.0
 + annotated-types==0.7.0
 + anyio==4.14.2
 + attrs==26.1.0
 + certifi==2026.7.22
 + charset-normalizer==3.4.9
 + distro==1.9.0
 + flatbuffers==25.12.19
 + frozenlist==1.8.0
 + greenlet==3.5.4
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + httpx-sse==0.4.3
 + idna==3.1

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [12]:
from langchain_community.document_loaders import TextLoader

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22700\2929458509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [20]:
loader = TextLoader("E:\My_Projects\Intelligent-Document-Assistant-with-OCR-RAG\data\Agentic_AI_RAG_Training.txt", encoding="utf8")
documents = loader.load()

print(documents[0].page_content)

Agentic AI

Agentic AI refers to AI systems that can plan, reason, use tools, and
take actions to achieve user-defined goals with limited supervision.
Unlike traditional AI models that simply respond to a single prompt,
agentic systems break complex tasks into smaller steps, decide what
information is needed, interact with external tools such as search
engines, databases, or APIs, and adapt based on intermediate results.

A typical Agentic AI workflow includes goal understanding, planning,
tool selection, execution, observation, and refinement. Large Language
Models (LLMs) often serve as the reasoning engine, while frameworks such
as LangChain or LangGraph coordinate tool usage and memory.
Retrieval-Augmented Generation (RAG) is commonly integrated so the agent
can retrieve relevant information from a vector database before
generating responses, reducing hallucinations and improving factual
accuracy.

Key components include: - Reasoning: deciding what to do next. - Memory:
retaining us

In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [24]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [25]:
text_chunks=text_splitter.split_documents(documents)

In [26]:
text_chunks

[Document(metadata={'source': 'E:\\My_Projects\\Intelligent-Document-Assistant-with-OCR-RAG\\data\\Agentic_AI_RAG_Training.txt'}, page_content='Agentic AI'),
 Document(metadata={'source': 'E:\\My_Projects\\Intelligent-Document-Assistant-with-OCR-RAG\\data\\Agentic_AI_RAG_Training.txt'}, page_content='Agentic AI refers to AI systems that can plan, reason, use tools, and\ntake actions to achieve user-defined goals with limited supervision.'),
 Document(metadata={'source': 'E:\\My_Projects\\Intelligent-Document-Assistant-with-OCR-RAG\\data\\Agentic_AI_RAG_Training.txt'}, page_content='Unlike traditional AI models that simply respond to a single prompt,\nagentic systems break complex tasks into smaller steps, decide what'),
 Document(metadata={'source': 'E:\\My_Projects\\Intelligent-Document-Assistant-with-OCR-RAG\\data\\Agentic_AI_RAG_Training.txt'}, page_content='information is needed, interact with external tools such as search\nengines, databases, or APIs, and adapt based on intermedia

In [27]:
! uv pip install faiss-cpu

Using Python 3.11.7 environment at: E:\My_Projects\Intelligent-Document-Assistant-with-OCR-RAG\.venv
Resolved 3 packages in 744ms
 Downloaded faiss-cpu
Prepared 1 package in 11.23s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 1.14s
 + faiss-cpu==1.14.3


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

ImportError: cannot import name 'OpenAIEmbeddings' from 'langchain.embeddings' (e:\My_Projects\Intelligent-Document-Assistant-with-OCR-RAG\.venv\Lib\site-packages\langchain\embeddings\__init__.py)

In [29]:
embeddings=OpenAIEmbeddings()

NameError: name 'OpenAIEmbeddings' is not defined

In [ ]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [ ]:
vectorstore

In [ ]:
retriever=vectorstore.as_retriever()

In [ ]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

In [ ]:
from langchain.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [ ]:
prompt=ChatPromptTemplate.from_template(template)

In [ ]:
prompt

In [ ]:
from langchain.schema.output_parser import StrOutputParser

In [ ]:
output_parser=StrOutputParser()

In [ ]:
from langchain.chat_models import ChatOpenAI

llm_model=ChatOpenAI(model_name="gpt-4o-mini")

In [ ]:
from langchain.schema.runnable import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [ ]:
rag_chain.invoke("tell me about Agentic AI")